# Capstone Project: Spam Detection using Deep Learning (NLP)

In [1]:
import os
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("team-ai/spam-text-message-classification")

print("Path to dataset files:", path)

csv_path = os.path.join(path, "SPAM text message 20170820 - Data.csv")
df = pd.read_csv(csv_path)
df.head()

c:\Users\singh\OneDrive\Desktop\capstone 2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 208k/208k [00:00<00:00, 225kB/s]

Extracting files...


Path to dataset files: C:\Users\singh\.cache\kagglehub\datasets\team-ai\spam-text-message-classification\versions\1


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
import re


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["clean_message"] = df["Message"].apply(clean_text)
df["label"] = df["Category"].map({"ham": 0, "spam": 1})
df[["Category", "label", "clean_message"]].head()

,Category,label,clean_message
0,ham,0,go until jurong point crazy available only in ...
1,ham,0,ok lar joking wif u oni
2,spam,1,free entry in a wkly comp to win fa cup final ...
3,ham,0,u dun say so early hor u c already then say
4,ham,0,nah i dont think he goes to usf he lives aroun...


In [3]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_message"], df["label"], test_size=0.20, random_state=42
)

vocab_size = 5000
max_len = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_train), maxlen=max_len, padding="post"
)
X_test_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_test), maxlen=max_len, padding="post"
)

print("Training Data Shape:", X_train_pad.shape)
print("Testing Data Shape:", X_test_pad.shape)

Training Data Shape: (4457, 100)
Testing Data Shape: (1115, 100)


In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

model = Sequential(
    [
        Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),
        GlobalAveragePooling1D(),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid"),
    ]
)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(
    X_train_pad, y_train, epochs=10, batch_size=32, validation_data=(X_test_pad, y_test)
)

c:\Users\singh\OneDrive\Desktop\capstone 2\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8600 - loss: 0.4189 - val_accuracy: 0.8664 - val_loss: 0.3763
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8658 - loss: 0.3747 - val_accuracy: 0.8664 - val_loss: 0.3724
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8658 - loss: 0.3698 - val_accuracy: 0.8664 - val_loss: 0.3665
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8658 - loss: 0.3595 - val_accuracy: 0.8664 - val_loss: 0.3526
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8658 - loss: 0.3363 - val_accuracy: 0.8664 - val_loss: 0.3169
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8670 - loss: 0.2909 - val_accuracy: 0.8682 - val_loss: 0.2606
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8950 - loss: 0.2247 - val_accuracy: 0.9202 - val_loss: 0.1897
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9477 - loss: 0.1576 - val_accuracy: 0.

In [5]:
loss, accuracy = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 97.67%


In [6]:
import pickle

model.save("spam_model.h5")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Saved spam_model.h5 and tokenizer.pkl successfully!")

Saved spam_model.h5 and tokenizer.pkl successfully!
